In [1]:
from src.tokenizers.BPE import BytePairEncoding
from src.train.lit_transformer import LitTransformer
from src.models.transformer import Transformer

# You need to instantiate the model used by LitTransformer and pass it in as the argument.
# Assume you know (or can specify) the Transformer architecture / args used at training.
# Here is an example (adjust paths/params as needed):

# Example: Load model config -- in production, load those from the checkpoint or your experiment config!
vocab_size = 37000          # Use actual vocab_size used for training
d_model = 256               # actual d_model
d_ff = 512                  # actual d_ff
n_heads = 8                 # actual n_heads
N = 6                       # actual N
seq_len = 256               # actual seq_len

# Create the underlying Transformer model -- adjust the imports and values as needed
transformer = Transformer(
    vocab_size=vocab_size,
    d_model=d_model,
    d_ff=d_ff,
    n_heads=n_heads,
    N=N,
    seq_len=seq_len,
)

transformer.eval()

# Now load the LitTransformer with the transformer argument
lit_model = LitTransformer.load_from_checkpoint(
    "/home/mcvjetko/phd/projects/transformer/experiments/en-hr/last.ckpt",
    transformer=transformer
)
transformer = lit_model.transformer
transformer.cuda()
bpe_tokenizer = BytePairEncoding.from_file("/home/mcvjetko/phd/projects/transformer/experiments/en-hr-tokenizers/tokenizer_37000.json")

In [2]:
# print(bpe_tokenizer._special_vocab)
# print(bpe_tokenizer.inv_vocab[4])
# print(bpe_tokenizer.inv_vocab[5])
# print(bpe_tokenizer.inv_vocab[6])
# print(bpe_tokenizer.inv_vocab[20257])
# print(bpe_tokenizer.inv_vocab[582])
# print(bpe_tokenizer.inv_vocab[661])
# print(bpe_tokenizer.inv_vocab[19172])

In [ ]:
import torch

raw = "This model is not good at translating dots."
text = bpe_tokenizer.tokenize(raw, max_length=256, add_special=True, pad=True)
#print("decoded", bpe_tokenizer.decode(text))
text = torch.tensor(text).to("cuda").unsqueeze(0)
text = text.repeat(2, 1)
#print(text)Julien is running often. -> Julien često trči.
# print(text.shape)
y = torch.full((1, 1), 1, dtype=torch.long).to("cuda")
y = y.repeat(2, 1)
print(text.shape, y.shape)
# print(y)
print(raw + " -> " + bpe_tokenizer.decode(transformer.translate(text, y)[1].tolist()))


torch.Size([2, 256]) torch.Size([2, 1])
Text. -> Teksas.


In [4]:
transformer.translate_beam_search(text, y)

x.shape torch.Size([2, 256])
y.shape torch.Size([2, 1])
tensor([[[-22.1366, -21.0878,  -8.2067,  ..., -26.0979, -17.9125, -21.4470]],

        [[-22.1366, -21.0878,  -8.2067,  ..., -26.0979, -17.9125, -21.4470]]],
       device='cuda:0')
initial topk vals tensor([[[-0.8635, -1.2647, -1.6759, -3.9675, -5.3537]],

        [[-0.8635, -1.2647, -1.6759, -3.9675, -5.3537]]], device='cuda:0')
initial topk vals shape torch.Size([2, 1, 5])
initial topk indices tensor([[[ 1480, 15308, 20711,    17,  2902]],

        [[ 1480, 15308, 20711,    17,  2902]]], device='cuda:0')
initial topk indices shape torch.Size([2, 1, 5])
torch.Size([2, 5, 1])
torch.Size([2, 5, 2])
y.shape torch.Size([2, 5, 2])
topkvals shape torch.Size([2, 1, 5])
logprobs torch.Size([2, 5])
next_token.shape: torch.Size([10, 37000])
token_logprobs.shape: torch.Size([10, 37000])
torch.Size([2, 5, 37000])
max token_logprobs per beam: tensor([[-1.2619, -0.6369, -0.1968, -1.0772, -0.3495],
        [-1.2619, -0.6369, -0.1968, -1.0772, 

/home/mcvjetko/phd/projects/transformer/src/models/transformer.py:331: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  token_logprobs = F.log_softmax(next_token)


tensor([[[    1, 20711,  1129,     2,     0,     0,     0,     0],
         [    1,  2902,   649,  1815,    18,     2,     0,     0],
         [    1, 15308,   571,     5,     2,     0,     0,     0],
         [    1,  2902,   649,  1815,  1957,     2,     0,     0],
         [    1,  2902,   649,  1815,   574,     4,  5144,     2]],

        [[    1, 20711,  1129,     2,     0,     0,     0,     0],
         [    1,  2902,   649,  1815,    18,     2,     0,     0],
         [    1, 15308,   571,     5,     2,     0,     0,     0],
         [    1,  2902,   649,  1815,  1957,     2,     0,     0],
         [    1,  2902,   649,  1815,   574,     4,  5144,     2]]],
       device='cuda:0')